# NPOV Drift Detection — Notebook 1: a dated ground-truth set of NPOV disputes

**Task:** [T428896 — WikiNPOV Drift Detector](https://phabricator.wikimedia.org/T428896)
**Notebook 1 of a planned series.** Read-only, English Wikipedia, public API only. No login, no edits, no ML.

## Why this notebook exists

The prototype built for T428896 implements nine signals and a changepoint detector, but its one real
validation run ([Phase 6](https://github.com/mishradev1/npov-drift-detector)) was a negative result:
a 100% false-positive rate on controls, and apparent "12–15 year lead times" that the repo itself
attributes to artifacts of an unthresholded detector. The stance signal's AUC ≈ 0.88 came from a
20-vs-20 hand-picked split, and its 0.93 accuracy came from a 70-probe set written by an LLM with a
single annotator and no inter-annotator agreement.

None of those numbers can move until there is a **dataset** they can be measured on. So we invert the
order of work: build and publish the evaluation set first, then test signals against it.

## What this notebook produces

1. **`npov_events_en.csv`** — one row per NPOV dispute, with the *exact revision that introduced the
   tag*, who added it, the edit summary, whether the tag is article-level or section-level, and which
   section. This is the event to be predicted, and the point lead time is measured from.
2. **`npov_controls_en.csv`** — topically matched articles with no neutrality tag in their history,
   matched on article length and revision count. Without these, a detector scores 100% recall by
   firing always.
3. **A measured label-noise figure** for the cheap "look for POV in the edit summary" heuristic that
   prior corpus work relies on, so later notebooks know how far to trust it.

## What this notebook does *not* claim

A `{{POV}}` tag marks the moment **a human noticed and cared enough to tag** — not the moment
neutrality actually degraded, and sometimes an ordinary content dispute rather than drift. That gap is
the central measurement problem of this project, not a footnote. It is discussed at the end and is the
subject of Notebook 2.

## Method

English Wikipedia's neutrality templates (`{{POV}}`, `{{NPOV}}`, `{{POV section}}`, `{{Unbalanced}}`, …)
take a `date=` parameter that files the article into a dated maintenance category, e.g.
`Category:Wikipedia neutral point of view disputes from February 2025`. At the time of writing there
are ~92 such categories covering July 2018 – October 2026 and ~2,176 articles.

The dated category gives us the *month*. To get the *revision* we:

1. list every revision in a window spanning the month before through the month after the stated month;
2. binary-search that window on "does this revision's wikitext carry a neutrality template"
   (≈ log₂ N content fetches, in practice 2–9);
3. return the first revision where the tag is present.

Narrowing by month is what makes this cheap, and what makes binary search's monotonicity assumption
safe — a tag is rarely added and removed twice inside three months. Every row records how many fetches
it took and whether the predecessor revision was confirmed tag-free, so labels stay auditable.

In [ ]:
# --- Configuration -------------------------------------------------------------
# Set this to your own Wikimedia username: API policy asks for a contactable
# User-Agent, and PAWS runs are attributed to you.
WIKI_USERNAME = "MahimaSinghal"

LANG        = "en"      # Notebook 5 revisits this for Indic wikis
N_POSITIVES = 120       # articles to localise. 120 ~ 15 min. None = all ~2176 (~5 h).
N_CONTROLS  = 120       # matched controls to build
# A control matched only on total length and lifetime revision count can still be dormant
# during the window a detector would actually see. In a pilot, 45% of such controls had
# zero edits in the 90 days before their matched positive's onset, against 15% of
# positives - so a model could separate the classes on "did anything happen at all".
# Requiring window activity removes that shortcut.
CONTROL_REQUIRE_ACTIVITY = True
CONTROL_WINDOW_DAYS      = 90
RANDOM_SEED = 20261017
CACHE_DIR   = "npov_cache"   # every API response is cached to disk, so re-runs are free
OUT_DIR     = "npov_data"

import os, re, json, time, math, random, hashlib, calendar, datetime
import requests
import pandas as pd

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
random.seed(RANDOM_SEED)

API = "https://" + LANG + ".wikipedia.org/w/api.php"
UA  = ("npov-drift-research/0.1 (https://phabricator.wikimedia.org/T428896; "
       "User:" + WIKI_USERNAME + ") python-requests")
print("API:", API)
print("UA :", UA)

In [ ]:
# --- Cached, rate-limited API client -------------------------------------------
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": UA})
_last_call = [0.0]
MIN_INTERVAL = 0.10          # <=10 req/s, well inside API etiquette guidance
api_stats = {"calls": 0, "cache_hits": 0, "bytes": 0}

def api(**params):
    # One API query, memoised on disk by its parameters.
    params.setdefault("action", "query")
    params.setdefault("format", "json")
    params.setdefault("formatversion", "2")
    key = hashlib.sha1(json.dumps(params, sort_keys=True, default=str).encode()).hexdigest()
    path = os.path.join(CACHE_DIR, key + ".json")
    if os.path.exists(path):
        api_stats["cache_hits"] += 1
        with open(path) as f:
            return json.load(f)
    wait = MIN_INTERVAL - (time.time() - _last_call[0])
    if wait > 0:
        time.sleep(wait)
    r = SESSION.get(API, params=params, timeout=60)
    _last_call[0] = time.time()
    r.raise_for_status()
    api_stats["calls"] += 1
    api_stats["bytes"] += len(r.content)
    data = r.json()
    if "error" in data:
        raise RuntimeError(data["error"])
    with open(path, "w") as f:
        json.dump(data, f)
    return data

def api_all(listkey, **params):
    # Follow continuation and concatenate a list= result.
    out, cont = [], {}
    while True:
        d = api(**params, **cont)
        out += d.get("query", {}).get(listkey, [])
        if "continue" not in d:
            return out
        cont = d["continue"]

print(api(titles="Wikipedia:Neutral point of view", prop="info")["query"]["pages"][0]["title"],
      "- API reachable")

## Step 1 — the positive pool

Enumerate the dated NPOV-dispute categories and their sizes. This is also the first check on whether
the tag population is usable at all: if nearly every dispute were from the last six months there would
be no history to learn drift from.

In [ ]:
MONTHS = {m: i for i, m in enumerate(calendar.month_name) if m}
DATED_PREFIX = "Wikipedia neutral point of view disputes from"

cats = api_all("allcategories", list="allcategories", acprefix=DATED_PREFIX,
               aclimit=500, acprop="size")

def cat_month(name):
    m = re.search(r"from (\w+) (\d{4})$", name)
    return datetime.date(int(m.group(2)), MONTHS[m.group(1)], 1) if m else None

def cat_name(c):
    return c.get("category") or c.get("*")      # formatversion 2 vs 1

pool = pd.DataFrame([{"category": "Category:" + cat_name(c),
                      "month": cat_month(cat_name(c)),
                      "n_pages": c.get("pages", 0)} for c in cats])
pool = pool.dropna(subset=["month"]).sort_values("month").reset_index(drop=True)

print(len(pool), "dated categories,", pool.n_pages.sum(), "tagged pages,",
      pool.month.min().strftime("%Y-%m"), "->", pool.month.max().strftime("%Y-%m"))
by_year = pool.assign(year=pool.month.map(lambda d: d.year)).groupby("year").n_pages.sum()
print("\nOpen disputes by year the tag was placed:")
print(by_year.to_string())

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.bar(range(len(pool)), pool.n_pages, width=0.9, color="#3366cc")
ticks = range(0, len(pool), 6)
ax.set_xticks(list(ticks))
ax.set_xticklabels([pool.month[i].strftime("%Y-%m") for i in ticks], rotation=45, ha="right")
ax.set_ylabel("open disputes")
ax.set_title("Currently open NPOV disputes on " + LANG + ".wikipedia, by month the tag was placed")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# Members of every dated category -> the positive pool.
rows = []
for _, c in pool.iterrows():
    for m in api_all("categorymembers", list="categorymembers", cmtitle=c.category,
                     cmnamespace=0, cmlimit=500):
        rows.append({"title": m["title"], "pageid": m["pageid"], "tag_month": c.month})

positives = pd.DataFrame(rows).drop_duplicates("pageid").reset_index(drop=True)
print(len(positives), "distinct tagged articles")
print("API calls so far:", api_stats["calls"], "(cache hits", api_stats["cache_hits"], ")")
positives.head()

## Step 2 — localising the tag to a single revision

The month is not enough: to measure lead time we need the timestamp of the edit that added the tag.

In [ ]:
# Neutrality templates whose presence we treat as an NPOV dispute marker.
# Deliberately narrow: {{Weasel}}, {{Peacock}}, {{Advert}} and {{COI}} are *style* and
# *conflict-of-interest* tags; lumping them in is one way prior work inflates its positives.
NPOV_TAGS = [
    "POV", "NPOV", "Neutrality", "POV-check", "POV check", "Disputed-section",
    "POV section", "POV-section", "POV lead", "POV-lead", "POV statement", "POV-statement",
    "POV title", "POV-title", "Unbalanced", "Unbalanced section", "Undue weight",
    "Undue-weight", "Neutrality disputed",
]
NPOV_TAG_RE = re.compile(r"\{\{\s*(" +
                         "|".join(re.escape(t).replace("\\ ", "[ _]") for t in NPOV_TAGS) +
                         r")\s*(\||\}\})", re.IGNORECASE)

def revisions_between(title, start, end):
    # Revision stubs, oldest first, with timestamp in [start, end].
    out, cont = [], {}
    while True:
        d = api(prop="revisions", titles=title,
                rvprop="ids|timestamp|user|comment|size", rvlimit=500, rvdir="newer",
                rvstart=start.isoformat() + "T00:00:00Z",
                rvend=end.isoformat() + "T23:59:59Z", **cont)
        page = d["query"]["pages"][0]
        out += page.get("revisions", [])
        if "continue" not in d:
            return out
        cont = d["continue"]

def wikitext(revid):
    d = api(prop="revisions", revids=revid, rvprop="content", rvslots="main")
    pages = d.get("query", {}).get("pages", [])
    if not pages or "revisions" not in pages[0]:
        return None                               # deleted or suppressed revision
    slot = pages[0]["revisions"][0]["slots"]["main"]
    return None if slot.get("texthidden") else slot.get("content")

def npov_tags(txt):
    # [(normalised template name, date= value or None)] for every neutrality template present
    out = []
    for m in NPOV_TAG_RE.finditer(txt):
        tail = txt[m.start(): m.start() + 300]
        dm = re.search(r"date\s*=\s*(\w+\s+\d{4})", tail)
        out.append((re.sub(r"[ _-]+", " ", m.group(1)).strip().lower(),
                    dm.group(1).strip() if dm else None))
    return out

def tagged(revid, want_month=None):
    # With want_month set, ONLY a template whose date= matches that month counts.
    # This matters more than it looks: an article can carry an old, long-resolved {{POV}}
    # alongside the newly dated tag, and matching "any neutrality tag" then dates the
    # dispute to the wrong decade. That single mistake left-censored 32% of a pilot run.
    txt = wikitext(revid)
    if txt is None:
        return None
    tags = npov_tags(txt)
    if want_month is None:
        return bool(tags)
    want = want_month.strftime("%B %Y").lower()
    return any(d and d.strip().lower() == want for _, d in tags)

def _bisect_onset(revs, pred):
    # (outcome, index, fetches) for the first revision in revs satisfying pred.
    lo, hi, fetches = 0, len(revs) - 1, 1
    if not pred(revs[hi]["revid"]):
        return "none_at_end", lo, fetches
    fetches += 1
    if pred(revs[lo]["revid"]):
        return "at_start", lo, fetches
    while lo + 1 < hi:
        mid = (lo + hi) // 2
        fetches += 1
        if pred(revs[mid]["revid"]):
            hi = mid
        else:
            lo = mid
    return "located", hi, fetches

def _month_offset(d, months):
    m = d.month - 1 + months
    return datetime.date(d.year + m // 12, m % 12 + 1, 1)

def find_onset(title, tag_month, slack_days=31, lookbacks=(1, 6, 24)):
    # First revision carrying the neutrality tag dated to this month.
    #
    # The lookback escalates. A tag dated "February 2025" is usually added in February,
    # but editors re-date tags and some tags are re-added after removal, so a one-month
    # lookback leaves a fifth of cases with the tag already present at the window start.
    # Widening is safe in strict mode, because we are hunting one specific dated template.
    eom = tag_month.replace(day=calendar.monthrange(tag_month.year, tag_month.month)[1])
    end = eom + datetime.timedelta(days=slack_days)
    rec = {"title": title, "tag_month": tag_month, "window_end": end,
           "fetches": 0, "status": None, "match_mode": None}
    revs, outcome = [], None

    for lb in lookbacks:
        start = _month_offset(tag_month, -lb)
        revs = revisions_between(title, start, end)
        rec.update({"window_start": start, "window_revs": len(revs), "lookback_months": lb})
        if not revs:
            continue
        outcome, idx, f = _bisect_onset(revs, lambda r: tagged(r, tag_month))
        rec["fetches"] += f
        rec["match_mode"] = "dated"
        if outcome == "located":
            return _fill_onset(rec, revs, idx)
        if outcome == "none_at_end":
            break              # the dated template is absent; try the lenient pass
        # "at_start": the tag predates this window, so widen and retry

    if not revs:
        rec["status"] = "no_revisions_in_window"
        return rec

    # lenient pass: the tag may have been added undated, with a bot filling in date= later
    outcome, idx, f = _bisect_onset(revs, lambda r: tagged(r, None))
    rec["fetches"] += f
    rec["match_mode"] = "any_tag"
    if outcome == "located":
        return _fill_onset(rec, revs, idx)
    rec["status"] = {"none_at_end": "untagged_at_window_end",
                     "at_start": "tagged_at_window_start"}[outcome]
    return rec

def _fill_onset(rec, revs, idx):
    # idx is the revision carrying the template dated to the category month. That is a
    # reliable anchor to the *right* dispute, but it is not always the human tagging:
    # a bot ("Dating maintenance tags: {{POV}}") adds date= to tags left undated, and
    # editors sometimes re-date. In a pilot run this hit 26% of rows. So when the
    # predecessor already carried a neutrality tag, walk back to the first revision in
    # the window that carries any tag, and treat *that* as the onset.
    anchor = revs[idx]
    rec["anchor_revid"] = anchor["revid"]
    rec["anchor_ts"] = anchor["timestamp"]
    rec["anchor_comment"] = anchor.get("comment") or ""
    onset, idx_o = anchor, idx
    rec["anchor_is_bot_dating"] = False
    rec["first_tag_censored"] = False

    prev_revid = revs[idx - 1]["revid"] if idx > 0 else None
    if prev_revid is not None and tagged(prev_revid, None):
        rec["anchor_is_bot_dating"] = True
        outcome_b, idx_b, f_b = _bisect_onset(revs[: idx + 1], lambda r: tagged(r, None))
        rec["fetches"] += f_b
        if outcome_b == "located":
            onset, idx_o = revs[idx_b], idx_b
        elif outcome_b == "at_start":
            # the tag was already there when the window opened: onset is left-censored,
            # so keep the anchor as a conservative upper bound and say so
            onset, idx_o = revs[0], 0
            rec["first_tag_censored"] = True

    rec["status"] = "located"
    rec["onset_revid"] = onset["revid"]
    rec["onset_ts"] = onset["timestamp"]
    rec["onset_user"] = onset.get("user")
    rec["onset_comment"] = onset.get("comment") or ""
    rec["onset_size"] = onset.get("size")
    rec["prev_revid"] = revs[idx_o - 1]["revid"] if idx_o > 0 else None
    rec["predecessor_clean"] = idx_o > 0 and not rec["first_tag_censored"]
    rec["anchor_lag_revs"] = idx - idx_o
    return rec

# smoke test on a case checked by hand
print(json.dumps(find_onset("Jews as the chosen people", datetime.date(2025, 5, 1)),
                 indent=1, default=str))

In [ ]:
# --- Localise a sample --------------------------------------------------------
sample = positives.sample(frac=1.0, random_state=RANDOM_SEED)
if N_POSITIVES:
    sample = sample.head(N_POSITIVES)
sample = sample.reset_index(drop=True)

records, t0 = [], time.time()
for i, row in sample.iterrows():
    try:
        records.append(find_onset(row.title, row.tag_month))
    except Exception as e:
        records.append({"title": row.title, "tag_month": row.tag_month,
                        "status": "error: " + type(e).__name__ + ": " + str(e)})
    if (i + 1) % 10 == 0 or i + 1 == len(sample):
        el = time.time() - t0
        print("  %d/%d  %5.0fs elapsed  (%.1f s/article, %d live calls)"
              % (i + 1, len(sample), el, el / (i + 1), api_stats["calls"]))

events = pd.DataFrame(records)
print("\nOutcome of localisation:")
print(events.status.value_counts().to_string())

In [ ]:
located = events[events.status == "located"].copy()
located["onset_ts"] = pd.to_datetime(located.onset_ts)

print("localised %d/%d = %.1f%%" % (len(located), len(events), 100 * len(located) / len(events)))
print("content fetches per article: median %.0f, max %.0f"
      % (located.fetches.median(), located.fetches.max()))
print("revisions in search window:  median %.0f, max %.0f"
      % (located.window_revs.median(), located.window_revs.max()))
print("\nhow the tag was matched (dated = exact template for this month):")
print(located.match_mode.value_counts().to_string())
print("\nlookback needed to find it (months before the stated month):")
print(located.lookback_months.value_counts().sort_index().to_string())

bot = located.anchor_is_bot_dating.fillna(False).astype(bool)
print("\nrows where the dated template was placed by a bot or re-dating edit, so the")
print("onset was walked back to the human tagging: %d (%.1f%%)" % (bot.sum(), 100 * bot.mean()))
print("  revisions walked back, median: %.0f" % located.loc[bot, "anchor_lag_revs"].median()
      if bot.any() else "  (none)")
cens = located.first_tag_censored.fillna(False).astype(bool)
print("rows still left-censored after the walk-back (onset predates the window): %d" % cens.sum())

# Does the located revision land inside the month the template claims? A strong sanity check.
in_month = located.apply(
    lambda r: (r.onset_ts.year, r.onset_ts.month) == (r.tag_month.year, r.tag_month.month), axis=1)
print("\nonset revision falls in the month stated by the template: %.1f%%" % (100 * in_month.mean()))
print("(mismatches are expected: a tag added on the 31st can be dated to the next month, and "
      "bots re-date undated tags)")

## Step 3 — scope and section

A section-level tag (`{{POV section}}`) and an article-level tag are different prediction problems: the
first names the paragraph to look at, the second does not. Prior work merges them, which makes the
"section embedding drift" null result hard to interpret. We separate them here.

In [ ]:
SECTION_TAGS = re.compile(r"^(POV[ _-]?section|POV[ _-]?lead|POV[ _-]?statement|"
                          r"Disputed-section|Unbalanced[ _]section)$", re.I)

def enrich_scope(revid, tag_month=None):
    # Which template fired, where in the page, and under which heading.
    txt = wikitext(revid)
    if txt is None:
        return {}
    want = tag_month.strftime("%B %Y").lower() if tag_month is not None else None
    chosen = None
    for m in NPOV_TAG_RE.finditer(txt):
        tail = txt[m.start(): m.start() + 300]
        dm = re.search(r"date\s*=\s*(\w+\s+\d{4})", tail)
        d = dm.group(1).strip() if dm else None
        if chosen is None:
            chosen = (m, d)
        if want and d and d.lower() == want:      # prefer the tag for *this* dispute
            chosen = (m, d)
            break
    if chosen is None:
        return {}
    m, tdate = chosen
    name = re.sub(r"[ _-]+", " ", m.group(1)).strip().lower()
    before = txt[:m.start()]
    headings = re.findall(r"^(={2,})\s*(.+?)\s*\1\s*$", before, re.M)
    is_section_template = bool(SECTION_TAGS.match(name.replace(" ", "")))
    scope = "section" if (is_section_template or headings) else "article"
    return {"template": name, "scope": scope,
            "section": headings[-1][1] if headings else None,
            "template_date": tdate,
            "article_chars": len(txt)}

ENRICH_COLS = ["template", "scope", "section", "template_date", "article_chars"]
enr = pd.DataFrame([enrich_scope(int(r.onset_revid), r.tag_month)
                    for _, r in located.iterrows()]).reindex(columns=ENRICH_COLS)
located = pd.concat([located.reset_index(drop=True), enr], axis=1)

print("Template used:")
print(located.template.value_counts().to_string())
print("\nScope:")
print(located.scope.value_counts(normalize=True).to_string())
print("\nArticle size at tagging: median {:,.0f} chars".format(located.article_chars.median()))
located[["title", "onset_ts", "template", "scope", "section", "onset_user"]].head(12)

## Step 4 — how noisy is the cheap heuristic?

Most prior NPOV-corpus work finds tag events by regexing **edit summaries** rather than diffing
wikitext, because it is far cheaper. Having just built the expensive ground truth, we can measure what
the shortcut costs. The number matters: if recall is high, later notebooks — and the Indic wikis,
where dated categories may not exist — can use the cheap path.

In [ ]:
COMMENT_RE = re.compile(
    r"\b(?:npov|pov|neutral|neutrality|unbalanc|undue|one[- ]sided|biased?)\b", re.I)

# Measure on the human tagging revisions only. Bot dating edits say
# "Dating maintenance tags: {{POV}}", which would flatter the heuristic badly.
clean = located[~located.first_tag_censored.fillna(False).astype(bool)]
hit = clean.onset_comment.fillna("").str.contains(COMMENT_RE)
print("edit summaries of human tag-adding revisions that mention neutrality: "
      "%.1f%%  (%d/%d)" % (100 * hit.mean(), hit.sum(), len(hit)))
print("so a summary-regex corpus builder would miss %.0f%% of real tag events"
      % (100 * (1 - hit.mean())))
bot_say = located.anchor_comment.fillna("").str.contains("Dating maintenance tags", case=False)
print("(for comparison: %d/%d anchor revisions are bot dating edits)"
      % (bot_say.sum(), len(located)))
print("\n--- examples the heuristic MISSES (silent taggings) ---")
for c in clean.loc[~hit, "onset_comment"].head(8):
    print("  .", (c or "<empty summary>")[:110])
print("\n--- examples it catches ---")
for c in clean.loc[hit, "onset_comment"].head(5):
    print("  .", c[:110])

In [ ]:
# A hand-auditable sample. Open these diffs and confirm the tag really appears.
audit = located.sample(min(8, len(located)), random_state=RANDOM_SEED)
print("Audit by eye - each diff should show a neutrality template being added:\n")
for _, r in audit.iterrows():
    print(r.title, " [" + str(r.template) + ", " + str(r.scope) + "]")
    print("   https://" + LANG + ".wikipedia.org/w/index.php?diff=" + str(int(r.onset_revid)))

## Step 5 — matched controls

A detector that fires on every article has perfect recall. The prototype's 100% control false-positive
rate is what happens when controls are not matched: NPOV-tagged articles are long, heavily edited and
in contested topic areas, so *any* activity-correlated signal separates them from random stubs — and
none of that is about neutrality.

So for each positive we draw a control from **one of its own topical categories**, require that no
neutrality tag appears in its current text or anywhere in its revision summaries, and keep the
candidate closest in (log length, log revision count).

In [ ]:
BAD_CAT = re.compile(r"(stub|births|deaths|\d{4}|Wikipedia|Articles|Pages|CS1|Category:All)", re.I)

def topical_categories(title):
    d = api(prop="categories", titles=title, cllimit=500, clshow="!hidden")
    pages = d["query"]["pages"]
    cats = [c["title"] for c in pages[0].get("categories", [])] if pages else []
    return [c for c in cats if not BAD_CAT.search(c)]

def page_stats(title):
    d = api(prop="info|revisions", titles=title, rvprop="ids", rvlimit=500, rvdir="newer")
    p = d["query"]["pages"][0]
    if p.get("missing"):
        return None
    return {"title": title, "pageid": p["pageid"], "length": p.get("length", 0),
            "n_revs_capped": len(p.get("revisions", [])),
            "revs_truncated": "continue" in d}

def never_npov_tagged(title, max_comment_calls=4):
    # No neutrality tag in the current text, and none mentioned in revision summaries.
    cur = api(prop="revisions", titles=title, rvprop="content", rvslots="main", rvlimit=1)
    pages = cur.get("query", {}).get("pages", [])
    if not pages or "revisions" not in pages[0]:
        return False
    txt = pages[0]["revisions"][0]["slots"]["main"].get("content") or ""
    if NPOV_TAG_RE.search(txt):
        return False
    cont, calls = {}, 0
    while calls < max_comment_calls:
        d = api(prop="revisions", titles=title, rvprop="comment", rvlimit=500,
                rvdir="newer", **cont)
        calls += 1
        for r in d["query"]["pages"][0].get("revisions", []):
            if COMMENT_RE.search(r.get("comment") or ""):
                return False
        if "continue" not in d:
            return True
        cont = d["continue"]
    return True        # screened as far as the call budget allows; flagged in the output

def active_in_window(title, end_ts, days=CONTROL_WINDOW_DAYS):
    """Did this article get edited at all in the days before end_ts?"""
    start = end_ts - datetime.timedelta(days=days)
    d = api(prop="revisions", titles=title, rvprop="ids", rvlimit=2, rvdir="newer",
            rvstart=start.strftime("%Y-%m-%dT%H:%M:%SZ"),
            rvend=end_ts.strftime("%Y-%m-%dT%H:%M:%SZ"))
    pages = d.get("query", {}).get("pages", [])
    return bool(pages) and len(pages[0].get("revisions", [])) > 0

def match_control(pos_row, n_candidates=8):
    cats_here = topical_categories(pos_row.title)
    if not cats_here:
        return None
    target = (math.log1p(pos_row.length or 1), math.log1p(pos_row.n_revs_capped or 1))
    for cat in random.sample(cats_here, k=min(3, len(cats_here))):
        members = api_all("categorymembers", list="categorymembers", cmtitle=cat,
                          cmnamespace=0, cmlimit=200)
        random.shuffle(members)
        best = None
        # Screens are ordered cheapest-first: one call for page stats, one for window
        # activity, and only then the summary scan, which can cost five. Reordering these
        # cut the cost of building a control set by roughly half.
        for m in members[:n_candidates]:
            if m["title"] == pos_row.title:
                continue
            st = page_stats(m["title"])                                    # 1 call
            if not st or st["length"] < 2000:
                continue
            if CONTROL_REQUIRE_ACTIVITY and not active_in_window(          # 1 call
                    m["title"], pd.Timestamp(pos_row.onset_ts).to_pydatetime()):
                continue
            if not never_npov_tagged(m["title"]):                          # up to 5 calls
                continue
            d = math.hypot(math.log1p(st["length"]) - target[0],
                           math.log1p(st["n_revs_capped"]) - target[1])
            if best is None or d < best[0]:
                best = (d, st, cat)
            if best[0] < 0.25:            # close enough; stop paying for more candidates
                break
        if best:
            return {**best[1], "matched_to": pos_row.title, "match_category": best[2],
                    "match_distance": round(best[0], 3)}
    return None

In [ ]:
# Stats for the positives, so controls can be matched against them
pos_stats = pd.DataFrame([s for s in (page_stats(t) for t in located.title) if s])
located = located.merge(pos_stats, on="title", how="left")

targets = located.dropna(subset=["length"]).head(N_CONTROLS)
controls, t0 = [], time.time()
for i, row in targets.iterrows():
    try:
        c = match_control(row)
    except Exception:
        c = None
    if c:
        controls.append(c)
    if (i + 1) % 10 == 0:
        print("  %d controls from %d attempts, %.0fs" % (len(controls), i + 1, time.time() - t0))

controls = pd.DataFrame(controls)
if len(controls):
    controls = controls.drop_duplicates("pageid")
print("\n%d matched controls for %d positives (%.0f%% match rate)"
      % (len(controls), len(targets), 100 * len(controls) / max(len(targets), 1)))

In [ ]:
# Did matching actually work? Compare the two distributions.
if len(controls):
    q = ["count", "25%", "50%", "75%"]
    print("Article length (chars):")
    print(pd.DataFrame({"positives": located.length.describe()[q],
                        "controls":  controls.length.describe()[q]}).to_string(), "\n")
    print("Revision count (capped at 500):")
    print(pd.DataFrame({"positives": located.n_revs_capped.describe()[q],
                        "controls":  controls.n_revs_capped.describe()[q]}).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
    for ax, col, lab in zip(axes, ["length", "n_revs_capped"],
                            ["article length (chars)", "revisions (capped 500)"]):
        ax.hist([located[col].dropna(), controls[col].dropna()], bins=18,
                label=["NPOV-tagged", "matched control"], color=["#cc3333", "#3366cc"])
        ax.set_xlabel(lab); ax.legend(frameon=False)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    axes[0].set_ylabel("articles")
    plt.suptitle("Matching check - these distributions should overlap heavily")
    plt.tight_layout(); plt.show()

In [ ]:
# --- Save ---------------------------------------------------------------------
ev_path = os.path.join(OUT_DIR, "npov_events_" + LANG + ".csv")
ct_path = os.path.join(OUT_DIR, "npov_controls_" + LANG + ".csv")
located.to_csv(ev_path, index=False)
controls.to_csv(ct_path, index=False)

meta = {
    "task": "https://phabricator.wikimedia.org/T428896",
    "notebook": "01_ground_truth",
    "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "wiki": LANG + ".wikipedia.org",
    "dated_categories": len(pool),
    "tagged_articles_available": int(pool.n_pages.sum()),
    "sampled": int(len(events)),
    "localised": int(len(located)),
    "controls": int(len(controls)),
    "random_seed": RANDOM_SEED,
    "live_api_calls": api_stats["calls"],
    "api_bytes": api_stats["bytes"],
    "npov_templates_matched": NPOV_TAGS,
}
with open(os.path.join(OUT_DIR, "run_metadata.json"), "w") as f:
    json.dump(meta, f, indent=2)

print(json.dumps(meta, indent=2))
print("\nwrote", ev_path, "\nwrote", ct_path)

## What we have, and what it is not

**We have** an auditable, reproducible event set: for each NPOV dispute, the revision that introduced
the tag, its scope and section, the tagger, and a topically matched control article. Everything comes
from the public API through an on-disk cache, so the run is cheap to repeat and every label can be
checked diff by diff.

**Known limitations**, roughly in order of how much they threaten later conclusions:

1. **The tag is a detection event, not an onset event.** The timestamp records when a human noticed.
   Real drift may precede it by years, or the tag may follow one bad edit with no drift at all. Any
   "lead time" reported later is lead time *over human attention* — the deployable quantity, but not
   the same as detecting true onset. Notebook 2 attacks this by reading the talk-page thread the
   tagger links to.
2. **Survivorship.** The dated categories hold only disputes that are *still open*, so this set leans
   towards unresolved and recent cases (2023 onward dominates). Resolved disputes need dumps or a
   comment-level history scan; the heuristic recall measured in Step 4 says how well the cheap scan
   would do.
3. **Left censoring.** Rows with status `tagged_at_window_start` had the tag before our window; they
   are excluded rather than silently mis-dated.
4. **Control screening is budget-limited.** `never_npov_tagged` scans up to 2,000 revision summaries
   per candidate, so a long-history article may carry an old, resolved dispute we never saw.
5. **One wiki, one language.** Everything here is English, and the dated-category trick needs
   re-verification per wiki. That is Notebook 5.

## Next: Notebook 2 — the null model

Before any stance or embedding signal, establish what *trivially* predicts an NPOV tag. On the set
above, fit the dullest possible baselines on edit metadata only — revision rate, revert rate, distinct
editor count, editor concentration (HHI), new-account share, length trajectory, pageviews, talk-page
activity — over the 90 days before the onset revision, with the same features computed over a matched
window for controls.

That gives the number every later signal has to beat, and it decides the shape of the project:

- if metadata alone separates tagged from control, the contribution has to be **lead time** and
  **cross-lingual transfer**, not detection;
- if it does not, the content signals have real room, and the Phase 6 negative result was a controls
  problem rather than a signal problem.

Either answer is worth publishing, which is the point of running it before building anything else.